In [1]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

google_model="gemini-3.1-flash-lite"

from langchain.chat_models import init_chat_model

llm = init_chat_model(model=google_model,model_provider="openai",
    openai_api_key=os.environ["GOOGLE_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

# **TOOLS**

## **DuckDuckGo Search Tool**

In [2]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool

@tool
def duckDuckGoSearchTool(query: str) -> str:
    """Searches the web using DuckDuckGo and returns the results."""
    search = DuckDuckGoSearchRun()
    return search.invoke(query)

/var/folders/j3/n04hhwvx3v960l20mpqqltsh0000gn/T/ipykernel_64217/2903240809.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


## **Arxiv Query Tool**

In [3]:
import arxiv
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

@tool
def arxivQueryTool(query: str) -> str:
    """Searches the arxiv for research data and returns the results."""
    api_wrapper = ArxivAPIWrapper(
        arxiv_search=arxiv.Search,
        arxiv_exceptions=(
            arxiv.ArxivError,
            arxiv.UnexpectedEmptyPageError,
            arxiv.HTTPError,
        ),
    )

    arxivQuery = ArxivQueryRun(api_wrapper=api_wrapper)
    return arxivQuery.invoke(query)


## **Wikipedia Query Tool**

In [4]:
import wikipedia
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

@tool
def wikiQueryTool(query: str) -> str:
    """ This tool uses the Wikipedia API to fetch information based on a query."""
    wikipedia.set_user_agent("MyLangGraphApp/1.0 (myemail@example.com)")
    wikiQuery = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=wikipedia))
    return wikiQuery.invoke(query)


## **Custom Tool**

In [5]:
from langchain.tools import tool

@tool
def personalInfoTool(name:str) -> str:

    """Returns personal information of the people"""
    info = {
        "Alice": "Alice is a software engineer with a passion for AI.",
        "Bob": "Bob is a data scientist who loves working with large datasets.",
        "Charlie": "Charlie is a product manager with experience in tech startups.",
    }
    return info.get(name, "Name not found.")

# **Tool Binding**

In [6]:
tools = [duckDuckGoSearchTool, arxivQueryTool, wikiQueryTool, personalInfoTool]

llm_with_tools = llm.bind_tools(tools=tools)

In [7]:
llm_with_tools.invoke("What is the info about Alice?")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 203, 'total_tokens': 219, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gemini-3.1-flash-lite', 'system_fingerprint': None, 'id': 'KldbavCzO7mDkdUP8vbA6A4', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f74cc-7f5e-72b1-b89d-4e2bdb77c794-0', tool_calls=[{'name': 'personalInfoTool', 'args': {'name': 'Alice'}, 'id': 'DKlGPleU', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 203, 'output_tokens': 16, 'total_tokens': 219, 'input_token_details': {}, 'output_token_details': {}})

In [8]:
response = llm_with_tools.invoke("What is the info about Alice?")
response.tool_calls

[{'name': 'personalInfoTool',
  'args': {'name': 'Alice'},
  'id': 'EWBpJEoO',
  'type': 'tool_call'}]

# **ReAct Agent**

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

# Initialize the Gemini model
model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)

# Correct way to create the LangGraph agent
agent = create_agent(model, tools=tools)


## **React Agent invocation with streams**

In [10]:
example_query = "What is the info about Alice?"

events = agent.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values"
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is the info about Alice?
================================== Ai Message ==================================

[]
Tool Calls:
  personalInfoTool (BVYvHMce)
 Call ID: BVYvHMce
  Args:
    name: Alice
================================= Tool Message =================================
Name: personalInfoTool

Alice is a software engineer with a passion for AI.
================================== Ai Message ==================================

[{'type': 'text', 'text': 'Alice is a software engineer with a passion for AI.', 'extras': {'signature': 'EjQKMgERTTIPzPgn9wRvCDpTXJVqpX0nEK1TAeKKUOwrd6ODpd9NAB2TGkCmB3WMmOG8SkqH'}}]
